In [ ]:
import pyproj
import svg
from geo_data import data_handler, helpers, svg_handler

In [ ]:
countries = data_handler.load("country", "ne", resolution=110)

In [ ]:
# center of europe
europe = countries[countries["continent"] == "Europe"]

exclude = ['RUS', 'ISL', 'TUR']  # exclude Russia, Iceland, Turkey
europe = europe[~europe['iso_a3'].isin(exclude)]

center = europe.union_all().centroid

In [ ]:
# orthographic projection
ortho_proj_str = f"+proj=ortho +lat_0={center.y} +lon_0={center.x}"
ortho_proj = pyproj.CRS.from_proj4(ortho_proj_str)

# project the countries
world_ortho = countries.to_crs(ortho_proj)

# Filter out polygons with inf coordinates
world_ortho = world_ortho[~world_ortho["geometry"].apply(data_handler.polygon_is_all_inf)]
world_ortho = world_ortho.sort_values("admin", ascending=False)

In [ ]:
svg_size = 500
assert ortho_proj.ellipsoid is not None
world_radius = ortho_proj.ellipsoid.semi_major_metre
canvas = svg_handler.MapSVG(svg_size, bounds=(-world_radius, world_radius))

# create definitions for radial gradient later
grad_id = "globeShadowGrad"
grad = svg_handler.get_radial_shadow_grad(grad_id)
defs = svg.Defs(elements=[grad])
canvas.add(defs)

# water background
globe_kwargs = {
    "cx": svg_size / 2,
    "cy": svg_size / 2,
    "r": svg_size / 2,
}
canvas.add(svg.Circle(id="sea", **globe_kwargs, **canvas.get_kwargs("sea")))

# countries
canvas.add_gdf(world_ortho, "countries", **canvas.get_kwargs("land"))

# radial gradient for globe shadow
canvas.add(svg.Circle(
    id="globeShadow",
    fill=f"url(#{grad_id})",
    **globe_kwargs  # type: ignore
))

# save
file_path = helpers.get_top_directory() / "results" / "europe_orthographic.svg"
canvas.save(file_path)
canvas